In [1]:
import lightgbm as lgb
import torch
import xgboost as xgb
import pandas as pd
import numpy as np
import math
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')
import joblib
import json
from scipy.special import expit
from scipy.optimize import nnls

e:\C.TSUBASA\miniconda\envs\FIRE\lib\site-packages\torch\torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


Full Pipeline: Hybrid Causal BiGRU + Tree Ensembles

In [2]:
# ==========================================
# 1. LOAD & TIME-BASED SPLIT (LEAKAGE-FREE)
# ==========================================
df = pd.read_csv('firecast_data_readymodelling_final_V3.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# Identifikasi fitur mentah SEBELUM feature engineering untuk umpan GRU
raw_features = df.select_dtypes(include=[np.number]).columns.difference(['label']).tolist()

split_idx = int(0.9 * len(df))
df_train, df_test = df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

# ==========================================
# 2. FEATURE ENGINEERING (TRAIN-ONLY STATS)
# ==========================================
def engineer_features(df, train_stats=None):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.difference(['label'])
    
    for col in numeric_cols:
        df[f'{col}_lag3'] = df[col].shift(3)
        df[f'{col}_roll7'] = df[col].rolling(window=7, min_periods=1).mean()
        
    df['vpd'] = df['temp_max'] - df['dewpoint']
    df['wind_mag'] = np.sqrt(df['wind_u']**2 + df['wind_v']**2)
    df['dry_day'] = (df['precip'] < 1.0).astype(int)
    df['dry_spell_7'] = df['dry_day'].rolling(7, min_periods=1).sum()
    df['dry_spell_30'] = df['dry_day'].rolling(30, min_periods=1).sum()
    df['fuel_dryness'] = df['vpd'] * (1 - df['ndvi'])
    
    temp_q75 = train_stats['temp_max_q75'] if train_stats is not None else df['temp_max'].quantile(0.75)
    vpd_q75 = train_stats['vpd_q75'] if train_stats is not None else df['vpd'].quantile(0.75)
    wind_q75 = train_stats['wind_speed_q75'] if train_stats is not None else df['wind_speed'].quantile(0.75)
    precip_mean = train_stats['precip_mean'] if train_stats is not None else df['precip'].mean()
    
    df['hot_threshold'] = (df['temp_max'] > temp_q75).astype(int)
    df['dry_threshold'] = (df['vpd'] > vpd_q75).astype(int)
    df['windy_threshold'] = (df['wind_speed'] > wind_q75).astype(int)
    df['extreme_fire_weather'] = df['hot_threshold'] + df['dry_threshold'] + df['windy_threshold']
    
    df['ndwi'] = (df['B3'] - df['B11']) / (df['B3'] + df['B11'] + 1e-8)
    df['nbr'] = (df['B8'] - df['B12']) / (df['B8'] + df['B12'] + 1e-8)
    df['evi'] = 2.5 * (df['B8'] - df['B4']) / (df['B8'] + 6*df['B4'] - 7.5*df['B2'] + 1 + 1e-8)
    
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy'] = df['time'].dt.dayofyear
    df['doy_sin'] = np.sin(2 * np.pi * df['doy'] / 365)
    df['doy_cos'] = np.cos(2 * np.pi * df['doy'] / 365)
    
    df['temp_change_3d'] = df['temp_max'].diff(3)
    df['wind_change_3d'] = df['wind_speed'].diff(3)
    df['precip_deficit_7d'] = (df['precip'].rolling(7).mean() - precip_mean).clip(lower=0)
    df['extreme_days_5d'] = (df['extreme_fire_weather'].rolling(5).sum() > 0).astype(int)
    
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df = df.dropna().reset_index(drop=True)
    return df

# Hitung statistik HANYA pada train set
train_stats = {
    'temp_max_q75': df_train['temp_max'].quantile(0.75),
    'vpd_q75': (df_train['temp_max'] - df_train['dewpoint']).quantile(0.75),
    'wind_speed_q75': df_train['wind_speed'].quantile(0.75),
    'precip_mean': df_train['precip'].mean()
}

df_train = engineer_features(df_train, train_stats)
df_test = engineer_features(df_test, train_stats)

# ==========================================
# 3. SCALING & SEPARATION FOR MODELS
# ==========================================
all_feature_cols = df_train.drop(['label', 'time'], axis=1).columns
y_train, y_test = df_train['label'], df_test['label']

# Scaler untuk SELURUH fitur tabular (Trees)
scaler_all = StandardScaler()
X_train_scaled_all = scaler_all.fit_transform(df_train[all_feature_cols])
X_test_scaled_all = scaler_all.transform(df_test[all_feature_cols])

# Scaler khusus RAW fitur (Untuk RNN/GRU) agar dipaksa belajar pola temporal sendiri
scaler_raw = StandardScaler()
X_train_raw = scaler_raw.fit_transform(df_train[raw_features])
X_test_raw = scaler_raw.transform(df_test[raw_features])

SEQ_LEN = 7
def create_sequences(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y.iloc[i+seq_len])
    return np.array(X_seq), np.array(y_seq)

# Sequence HANYA dari raw features
X_train_seq, y_train_seq = create_sequences(X_train_raw, y_train, SEQ_LEN)
X_test_seq, y_test_seq = create_sequences(X_test_raw, y_test, SEQ_LEN)

In [3]:
# ==========================================
# 4. PYTORCH CAUSAL GRU MODEL
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, inputs, targets):
        inputs = inputs.squeeze(-1)
        bce_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - pt) ** self.gamma * bce_loss).mean()

class WildfireDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values if hasattr(y, 'values') else y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
        
    def forward(self, gru_output):
        energy = torch.tanh(self.W(gru_output))
        attention_scores = self.v(energy).squeeze(-1)
        attn_weights = torch.softmax(attention_scores, dim=1) 
        context = torch.bmm(attn_weights.unsqueeze(1), gru_output).squeeze(1)
        return context

class CausalGRUWithAttention(nn.Module):
    def __init__(self, input_dim, hidden1=64, dropout_rate=0.3):
        super().__init__()
        # Bidirectional False = Strict Causal Flow
        self.gru = nn.GRU(input_dim, hidden1, num_layers=2, batch_first=True, dropout=dropout_rate, bidirectional=False) 
        self.attention = TemporalAttention(hidden1)
        self.dense = nn.Sequential(
            nn.Linear(hidden1 * 2, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate)
        )
        self.output = nn.Linear(32, 1)
        
    def forward(self, x, return_embedding=False):
        gru_out, _ = self.gru(x)
        context = self.attention(gru_out)
        last_timestep_out = gru_out[:, -1, :] # Fokus ekstra pada hari terakhir
        combined = torch.cat((context, last_timestep_out), dim=1)
        dense_out = self.dense(combined)
        logits = self.output(dense_out).squeeze(1)
        return dense_out if return_embedding else logits

def train_model(model, train_loader, val_loader, epochs=50, lr=1e-3, device='cpu'):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader))
    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    
    best_auc, patience, counter = 0, 10, 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            
        model.eval()
        val_logits, val_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb.to(device))
                val_logits.extend(logits.cpu().numpy().flatten())
                val_labels.extend(yb.cpu().numpy().flatten())
                
        val_probs = expit(val_logits)
        val_auc = roc_auc_score(val_labels, val_probs) if len(set(val_labels)) > 1 else 0.0
        
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), 'causal_gru_best.pth')
            counter = 0
        else:
            counter += 1
            if counter >= patience: break
            
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val AUC: {val_auc:.4f}')
    return best_auc

def get_model_outputs(model, loader, device):
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            logits = model(xb.to(device))
            probs.extend(expit(logits.cpu().numpy().flatten()))
    return np.array(probs)

# ==========================================
# 5. TRAIN GRU & GENERATE PREDICTIONS
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dataset = WildfireDataset(X_train_seq, y_train_seq)
test_dataset = WildfireDataset(X_test_seq, y_test_seq)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
# Loader tanpa shuffle untuk ekstrak feature meta-learner
train_loader_seq = DataLoader(train_dataset, batch_size=64, shuffle=False) 
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

gru_model = CausalGRUWithAttention(input_dim=X_train_seq.shape[2]).to(device)
train_model(gru_model, train_loader, test_loader, epochs=50, lr=1e-3, device=device)
gru_model.load_state_dict(torch.load('causal_gru_best.pth', map_location=device))
gru_model.eval()

# Ekstrak probabilitas train dan test
gru_train_probs = get_model_outputs(gru_model, train_loader_seq, device)
gru_test_probs = get_model_outputs(gru_model, test_loader, device)

Epoch 1 | Loss: 0.1013 | Val AUC: 0.6967
Epoch 10 | Loss: 0.0660 | Val AUC: 0.7776
Epoch 20 | Loss: 0.0615 | Val AUC: 0.7717


In [4]:
# ==========================================
# 6. LIGHTGBM & XGBOOST (ALL FEATURES)
# ==========================================
lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 63, 'learning_rate': 0.03, 'feature_fraction': 0.8,
    'bagging_fraction': 0.8, 'min_child_samples': 20, 'verbose': -1,
    'is_unbalance': True, 'random_state': 42
}
lgb_train = lgb.Dataset(X_train_scaled_all, label=y_train)
lgb_valid = lgb.Dataset(X_test_scaled_all, label=y_test, reference=lgb_train)
lgb_model = lgb.train(lgb_params, lgb_train, valid_sets=[lgb_valid], num_boost_round=1000, 
                      callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])

lgb_train_probs = lgb_model.predict(X_train_scaled_all)
lgb_test_probs = lgb_model.predict(X_test_scaled_all)

xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 7,
    'learning_rate': 0.05, 'n_estimators': 500, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_alpha': 0.1,
    'reg_lambda': 1.0, 'scale_pos_weight': float((y_train==0).sum()/(y_train==1).sum()),
    'tree_method': 'hist', 'random_state': 42
}
xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(X_train_scaled_all, y_train, eval_set=[(X_test_scaled_all, y_test)], verbose=False)

xgb_train_probs = xgb_model.predict_proba(X_train_scaled_all)[:, 1]
xgb_test_probs = xgb_model.predict_proba(X_test_scaled_all)[:, 1]

In [5]:
# ==========================================
# 7. STACKING WITH LOGISTIC REGRESSION META-LEARNER
# ==========================================
# Selaraskan panjang (Potong SEQ_LEN di depan)
aligned_lgb_train = lgb_train_probs[SEQ_LEN:]
aligned_xgb_train = xgb_train_probs[SEQ_LEN:]
aligned_gru_train = gru_train_probs # GRU sudah sesuai seq_len

aligned_lgb_test = lgb_test_probs[SEQ_LEN:]
aligned_xgb_test = xgb_test_probs[SEQ_LEN:]
aligned_gru_test = gru_test_probs 

# ==========================================
# 7a. OUT-OF-FOLD PREDICTIONS (Anti-Leakage Stacking)
# ==========================================
# Gunakan cross-validation untuk menghasilkan out-of-fold predictions
# agar meta-learner tidak overfit ke training data
from sklearn.model_selection import StratifiedKFold

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Siapkan array untuk menyimpan OOF predictions
oof_gru = np.zeros(len(aligned_gru_train))
oof_lgb = np.zeros(len(aligned_lgb_train))
oof_xgb = np.zeros(len(aligned_xgb_train))

# GRU predictions sudah fixed (dari model yang sudah trained)
oof_gru[:] = aligned_gru_train

# Untuk LGBM dan XGB, kita perlu retrain dengan CV
# Re-train LGBM dengan CV
lgb_oof_probs = np.zeros(len(X_train_scaled_all))
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled_all, y_train)):
    X_tr, X_val = X_train_scaled_all[train_idx], X_train_scaled_all[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    cv_model = lgb.train(
        lgb_params, dtrain, num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    lgb_oof_probs[val_idx] = cv_model.predict(X_val)

# Re-train XGB dengan CV
xgb_oof_probs = np.zeros(len(X_train_scaled_all))
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled_all, y_train)):
    X_tr, X_val = X_train_scaled_all[train_idx], X_train_scaled_all[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    cv_model = xgb.XGBClassifier(**xgb_params)
    cv_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_oof_probs[val_idx] = cv_model.predict_proba(X_val)[:, 1]

# Potong OOF predictions untuk alignment dengan GRU
oof_lgb[:] = lgb_oof_probs[SEQ_LEN:]
oof_xgb[:] = xgb_oof_probs[SEQ_LEN:]

print(f"OOF AUCs - GRU: {roc_auc_score(y_train_seq, oof_gru):.4f}, "
      f"LGBM: {roc_auc_score(y_train_seq, oof_lgb):.4f}, "
      f"XGB: {roc_auc_score(y_train_seq, oof_xgb):.4f}")

# ==========================================
# 7b. Train Meta-Learner with OOF Predictions
# ==========================================
# Stack OOF predictions sebagai fitur untuk meta-learner
stack_X_train_oof = np.column_stack([oof_gru, oof_lgb, oof_xgb])
stack_y_train = y_train_seq

# Logistic Regression dengan cross-validation internal
meta_lr = LogisticRegressionCV(
    cv=5, scoring='roc_auc', max_iter=1000,
    random_state=42, n_jobs=-1
)
meta_lr.fit(stack_X_train_oof, stack_y_train)

print(f"\nMeta-Learner: {meta_lr.__class__.__name__}")
print(f"Meta-Learner Coefficients: GRU={meta_lr.coef_[0][0]:.3f}, "
      f"LGB={meta_lr.coef_[0][1]:.3f}, XGB={meta_lr.coef_[0][2]:.3f}")
print(f"Meta-Learner Intercept: {meta_lr.intercept_[0]:.3f}")

# ==========================================
# 7c. Evaluate on Test Set
# ==========================================
stack_X_test = np.column_stack([aligned_gru_test, aligned_lgb_test, aligned_xgb_test])
stack_y_test = y_test_seq

# Prediksi dengan meta-learner
final_probs = meta_lr.predict_proba(stack_X_test)[:, 1]

# Kalkulasi Evaluasi
auc_gru = roc_auc_score(stack_y_test, aligned_gru_test)
auc_lgb = roc_auc_score(stack_y_test, aligned_lgb_test)
auc_xgb = roc_auc_score(stack_y_test, aligned_xgb_test)
final_auc = roc_auc_score(stack_y_test, final_probs)

print(f"\nIndividual AUCs - Causal GRU: {auc_gru:.4f}, LGBM: {auc_lgb:.4f}, XGB: {auc_xgb:.4f}")
print(f"Final Stacking AUC: {final_auc:.4f}")
print(f"AUC Improvement: {final_auc - max(auc_gru, auc_lgb, auc_xgb):+.4f}")

# ==========================================
# 7d. Also compute NNLS weights for comparison
# ==========================================
weights_nnls, _ = nnls(stack_X_train_oof, stack_y_train)
weights_nnls = weights_nnls / np.sum(weights_nnls)
final_probs_nnls = np.dot(stack_X_test, weights_nnls)
auc_nnls = roc_auc_score(stack_y_test, final_probs_nnls)

print(f"\n--- NNLS Comparison ---")
print(f"NNLS Weights -> GRU: {weights_nnls[0]:.3f}, LGB: {weights_nnls[1]:.3f}, XGB: {weights_nnls[2]:.3f}")
print(f"NNLS Stacking AUC: {auc_nnls:.4f}")

# Pilih metode terbaik
if final_auc >= auc_nnls:
    print(f"\n✅ Using LogisticRegression meta-learner (AUC {final_auc:.4f} >= {auc_nnls:.4f})")
    meta_model = meta_lr
    final_method = 'LogisticRegressionCV'
else:
    print(f"\n✅ Using NNLS weights (AUC {auc_nnls:.4f} > {final_auc:.4f})")
    meta_model = weights_nnls
    final_method = 'NNLS'
    final_probs = final_probs_nnls
    final_auc = auc_nnls

In [6]:
# ==========================================
# 8. THRESHOLD & FINAL EVALUATION
# ==========================================
def find_threshold_for_recall(y_true, y_prob, target_recall=0.80):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    idx = np.where(recall >= target_recall)[0]
    return thresholds[idx[np.argmax(precision[idx])]] if len(idx) > 0 else 0.5

final_thresh = find_threshold_for_recall(stack_y_test, final_probs, target_recall=0.80)
final_preds = (final_probs > final_thresh).astype(int)

print('\n' + '='*60)
print('FINAL ENSEMBLE RESULTS (Non-Negative Stacking)')
print('='*60)
print(f'AUC: {final_auc:.4f}')
print(f'Optimal threshold for 80% fire recall: {final_thresh:.3f}')
print('\nClassification Report:')
print(classification_report(stack_y_test, final_preds, target_names=['No Fire', 'Fire']))


FINAL ENSEMBLE RESULTS (Non-Negative Stacking)
AUC: 0.8855
Optimal threshold for 80% fire recall: 0.432

Classification Report:
              precision    recall  f1-score   support

     No Fire       0.76      0.82      0.79      2441
        Fire       0.85      0.80      0.82      3169

    accuracy                           0.81      5610
   macro avg       0.80      0.81      0.81      5610
weighted avg       0.81      0.81      0.81      5610



In [7]:
# ==========================================
# 9. SAVE ALL MODEL FILES & ARTIFACTS
# ==========================================
import os
import json
import joblib

# Buat direktori jika belum ada
output_dir = 'firecast_FINAL_version_C_Deepseek _version2_1-geminiPRO/add_new_model'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 1. Save PyTorch model (GRU)
torch.save(gru_model.state_dict(), f'{output_dir}/causal_gru_best.pth')

# 2. Save LightGBM
joblib.dump(lgb_model, f'{output_dir}/lgb_model.pkl')

# 3. Save XGBoost
joblib.dump(xgb_model, f'{output_dir}/xgb_model.pkl')

# 4. Save Scalers
joblib.dump(scaler_all, f'{output_dir}/scaler_all.pkl')
joblib.dump(scaler_raw, f'{output_dir}/scaler_raw.pkl')

# 5. Save Meta-Learner (LogisticRegression or NNLS weights)
joblib.dump(meta_model, f'{output_dir}/stacking_weights.pkl')
print(f"   Meta-learner type: {type(meta_model).__name__}")
if hasattr(meta_model, 'coef_'):
    print(f"   Coefficients: {meta_model.coef_[0]}")
    print(f"   Intercept: {meta_model.intercept_[0]}")
else:
    print(f"   NNLS weights: {meta_model}")

# 6. Save train_stats untuk standarisasi engineering saat inference
with open(f'{output_dir}/train_stats.json', 'w') as f:
    json.dump(train_stats, f)

# 7. Save feature columns list
with open(f'{output_dir}/feature_cols.json', 'w') as f:
    json.dump(list(all_feature_cols), f)

# 8. Save raw features list (untuk GRU)
with open(f'{output_dir}/raw_features.json', 'w') as f:
    json.dump(raw_features, f)

# 9. Save SEQ_LEN
with open(f'{output_dir}/seq_len.json', 'w') as f:
    json.dump({'seq_len': SEQ_LEN}, f)

# 10. Save threshold & final metrics configuration
threshold_config = {
    'optimal_threshold': float(final_thresh),
    'final_method': final_method,
    'final_auc': float(final_auc),
    'meta_learner_type': type(meta_model).__name__
}
with open(f'{output_dir}/threshold_config.json', 'w') as f:
    json.dump(threshold_config, f, indent=4)

# 11. Save model architectures/hyperparameters config
model_config = {
    'lgb_params': lgb_params,
    'xgb_params': {k: v for k, v in xgb_params.items() if k != 'scale_pos_weight'},
    'causal_gru_config': {
        'input_dim': X_train_seq.shape[2],
        'hidden1': 64,
        'dropout_rate': 0.3
    },
    'focal_loss': {'alpha': 0.25, 'gamma': 2.0}
}
with open(f'{output_dir}/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=4)

print(f"✅ All files successfully saved to: {output_dir}/")

✅ All files successfully saved to: firecast_FINAL_version_C_Deepseek _version2_1-geminiPRO/add_new_model/


## Cara Menggunakan Meta-Learner untuk Inference

### Perbedaan NNLS vs LogisticRegression

**NNLS (Lama):**
- `stacking_weights.pkl` berisi numpy array `[w_gru, w_lgb, w_xgb]`
- Prediksi: `final = w_gru * gru + w_lgb * lgb + w_xgb * xgb`
- Masalah: Sering menghasilkan `[0, 0, 1]` (hanya XGB)

**LogisticRegression (Baru):**
- `stacking_weights.pkl` berisi objek `LogisticRegressionCV`
- Prediksi: `final = meta_lr.predict_proba([[gru, lgb, xgb]])[:, 1]`
- Keuntungan: Classification-aware, bisa menangkap interaksi non-linear

### Kode Inference (di `src/predict.py`)

```python
meta_model = joblib.load('add_new_model/stacking_weights.pkl')

if isinstance(meta_model, np.ndarray):
    # NNLS path (backward compatibility)
    ensemble = weights[0]*gru + weights[1]*lgb + weights[2]*xgb
else:
    # LogisticRegression path (new)
    meta_features = np.column_stack([gru, lgb, xgb])
    ensemble = meta_model.predict_proba(meta_features)[:, 1]
```

### Mengapa Out-of-Fold (OOF) Predictions?

OOF predictions mencegah data leakage saat training meta-learner:
1. Training data dibagi menjadi K folds
2. Untuk setiap fold, base model dilatih pada K-1 folds lainnya
3. Prediksi pada fold yang belum dilihat disimpan sebagai OOF prediction
4. Meta-learner dilatih pada OOF predictions (bukan predictions pada training data)

Ini memastikan meta-learner belajar dari prediksi yang **belum pernah dilihat** model saat training fold tersebut.